# Q-HPC — Step 6/7 Calibration on Colab

Calibrates the resource profiles in `backend/utils/resource_constants.py`
against real measurements, and measures a real CUDA speedup factor.

**Why Colab:** it is Linux with a real NVIDIA GPU, so scikit-learn, qiskit,
qiskit-aer and xgboost all work, and the CUDA code paths can actually be
timed. A Windows host with Smart App Control blocks several of these
packages' compiled extensions, and a laptop without an NVIDIA GPU cannot
measure a GPU speedup at all.

---

## Read this before trusting any number below

The three fitted quantities transfer between machines **unequally**:

| Quantity | Transfers to your laptop? | Why |
|---|---|---|
| `memory_exp`, `runtime_exp` | **Yes** | Complexity class is a property of the algorithm |
| `base_memory_gb` | **Broadly** | An algorithm allocates roughly what it allocates |
| `base_runtime_s` | **No** | Colab's Xeon is not your CPU |
| `GPU_SPEEDUP_FACTOR` | **No** | Measured against Colab's CPU and its specific GPU |

So: take the **exponents** and **memory** from here with confidence, take
**runtime** figures as Colab's own, and record the host either way. Every
emitted block below includes a `calibration_host` field for exactly this
reason. In the write-up, a runtime figure without a named host is not a
reportable result.

**First:** Runtime → Change runtime type → T4 GPU → Save.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

`qiskit-aer-gpu` replaces `qiskit-aer` — it carries the CUDA statevector
backend. Installing both causes an import conflict, so the CPU wheel is
removed first.

In [ ]:
!pip install -q pydantic psutil scikit-learn scipy numpy
!pip install -q xgboost
!pip uninstall -q -y qiskit-aer
!pip install -q qiskit qiskit-aer-gpu
print("done")

In [ ]:
# Verify everything imports — report the real error if not.
for name in ("numpy", "scipy", "sklearn", "xgboost", "qiskit", "qiskit_aer",
             "pydantic", "psutil"):
    try:
        mod = __import__(name)
        print(f"  OK    {name:<12} {getattr(mod, '__version__', '?')}")
    except Exception as exc:
        print(f"  FAIL  {name:<12} {type(exc).__name__}: {exc}")

## 3. Get the Q-HPC code

Two options. Use the clone if `backend/models/` is in the repo by now;
otherwise upload a zip of your local `backend/` folder.

In [ ]:
# OPTION A — clone the repo
!git clone -b dev https://github.com/ronsmessi1012/QHPC.git /content/QHPC
%cd /content/QHPC
!ls backend

In [ ]:
# OPTION B — upload a zip of your local backend/ folder instead.
# Run this ONLY if Option A is missing backend/models/.
#
# from google.colab import files
# import zipfile, os
# os.makedirs("/content/QHPC", exist_ok=True)
# up = files.upload()                       # select backend.zip
# for fn in up:
#     with zipfile.ZipFile(fn) as z:
#         z.extractall("/content/QHPC")
# %cd /content/QHPC
# !ls backend

## 4. What does the framework detect here?

This is `backend/utils/hardware_detect.py` running unmodified. On Colab it
should find the T4; on your laptop the same code reports no CUDA device.
That difference is the point — the mapper assigns only what is present.

In [ ]:
!python -m backend.utils.hardware_detect

## 5. Calibrate the classical profiles

`--repeats 3` takes the median of three runs per size, which matters for
peak-memory sampling. Expect a few minutes; SVM at larger sizes is O(n²).

In [ ]:
!python -m backend.tests.benchmark_resources --repeats 3 --json /content/calibration_colab.json

## 6. Measure the real CUDA speedup

`GPU_SPEEDUP_FACTOR = 6.0` in `resource_constants.py` is a placeholder I
had no way to verify. This measures it for XGBoost: identical data and
hyperparameters, `device="cpu"` versus `device="cuda"`.

Note what this ratio is and is not. It is XGBoost's speedup on *this* GPU
against *Colab's* CPU. It is not a universal constant, and it is not what
you would see on another machine.

In [ ]:
import time
import numpy as np
import xgboost as xgb

def bench_xgb(n, device, repeats=3):
    rng = np.random.default_rng(0)
    X = rng.standard_normal((n, 50))
    y = rng.integers(0, 2, n)
    times = []
    for _ in range(repeats):
        model = xgb.XGBClassifier(n_estimators=200, max_depth=6,
                                  tree_method="hist", device=device,
                                  verbosity=0)
        t0 = time.perf_counter()
        model.fit(X, y)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

print(f"{'n':>9}  {'CPU (s)':>9}  {'GPU (s)':>9}  {'speedup':>8}")
print("-" * 42)
ratios = []
for n in (10_000, 50_000, 200_000, 500_000):
    t_cpu = bench_xgb(n, "cpu")
    t_gpu = bench_xgb(n, "cuda")
    ratio = t_cpu / t_gpu
    ratios.append((n, ratio))
    print(f"{n:>9,}  {t_cpu:>9.3f}  {t_gpu:>9.3f}  {ratio:>7.2f}x")

print()
print("GPU speedup is size-dependent: kernel-launch and transfer overhead")
print("dominates on small inputs, so a single scalar factor is a")
print("simplification. Report the size it was measured at.")
big = [r for n, r in ratios if n >= 200_000]
if big:
    print(f"\nSuggested GPU_SPEEDUP_FACTOR (large-input regime): "
          f"{np.median(big):.1f}")

## 7. Measure quantum statevector simulation

Two things at once:

1. **`SIM_FLOPS_PER_SECOND`** — the crudest constant in the whole file.
   It converts gate counts to seconds and was a pure guess at `1e9`.
2. **GPU vs CPU statevector simulation** — a genuine result worth reporting
   in its own right, independent of the mapper.

The memory formula needs no calibration: statevector memory is exactly
`2^n × 16` bytes, which is arithmetic, not an empirical question.

In [ ]:
import time
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def build_circuit(n, layers=3):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for _ in range(layers):
        for q in range(0, n - 1, 2):
            qc.cx(q, q + 1)
        for q in range(1, n - 1, 2):
            qc.cx(q, q + 1)
    qc.save_statevector()
    return qc

def two_qubit_gate_count(n, layers=3):
    # Same gate model the estimator uses: depth * n / 2
    depth = layers * 2
    return max(1, (depth * n) // 2)

def bench_sim(n, device, repeats=3):
    sim = AerSimulator(method="statevector", device=device)
    compiled = transpile(build_circuit(n), sim)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        sim.run(compiled, shots=1).result()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

print(f"{'qubits':>7}  {'mem (GB)':>9}  {'CPU (s)':>9}  {'GPU (s)':>9}  "
      f"{'speedup':>8}  {'CPU flops/s':>12}")
print("-" * 68)

flops = []
for n in (16, 20, 22, 24, 26):
    mem_gb = (2 ** n) * 16 / (1024 ** 3)
    t_cpu = bench_sim(n, "CPU")
    try:
        t_gpu = bench_sim(n, "GPU")
        speed = f"{t_cpu / t_gpu:>7.2f}x"
    except Exception as exc:
        t_gpu, speed = float("nan"), f"  n/a"
    f = two_qubit_gate_count(n) * (2 ** n) / t_cpu
    flops.append(f)
    print(f"{n:>7}  {mem_gb:>9.4f}  {t_cpu:>9.3f}  {t_gpu:>9.3f}  "
          f"{speed}  {f:>12.3g}")

print()
print(f"Suggested SIM_FLOPS_PER_SECOND (median, CPU): {np.median(flops):.3g}")
print(f"Current placeholder in resource_constants.py: 1e9")

## 8. Verify the memory formula

The estimator claims statevector memory is exactly `2^n × 16` bytes. This
checks that against the real allocated statevector rather than taking it on
trust.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

for n in (10, 14, 18):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.save_statevector()
    sim = AerSimulator(method="statevector", device="CPU")
    sv = sim.run(transpile(qc, sim), shots=1).result().get_statevector()
    arr = np.asarray(sv)
    actual = arr.nbytes / (1024 ** 3)
    predicted = (2 ** n) * 16 / (1024 ** 3)
    print(f"n={n:>3}  amplitudes={arr.size:>9,}  dtype={arr.dtype}  "
          f"actual={actual:.8f} GB  predicted={predicted:.8f} GB  "
          f"match={np.isclose(actual, predicted)}")

## 9. Confirm the mapper behaves differently here

Same code, same workload, different hardware. On Colab the GPU exists, so
a GPU-capable algorithm with a large footprint maps to `gpu_cuda`. On your
laptop the identical call maps to `cpu` and says why.

This cell needs `backend/models/` to be present.

In [ ]:
from backend.models.problem_schema import ProblemSpecification
from backend.models.decision_schema import ExecutionDecision
from backend.agents.resource_estimator import estimate_resources
from backend.engine.hardware_mapper import map_hardware
from backend.utils.hardware_detect import detect_hardware

inv = detect_hardware()
print(f"planning mode : {inv.planning_mode.value}")
print(f"CUDA          : {inv.gpu_description}")
print(f"MPI           : {'yes, %d slots' % inv.mpi_slots if inv.mpi_available else 'no'}")
print()

cases = [
    ("quantum", "Quantum Approximate Optimization Algorithm (QAOA)", 500, 500),
    ("classical", "XGBoost", 100, 2_000_000),
    ("classical", "Genetic Algorithm", 500, 500),
    ("quantum", "Grover Search", 1000, 1000),
]

for family, algorithm, dim, ds in cases:
    spec = ProblemSpecification(
        user_query="calibration probe", problem_class="optimization",
        confidence=0.8, dimensionality=dim, dataset_size=ds,
        constraint_structure="constrained")
    decision = ExecutionDecision(
        selected_family=family, selected_algorithm=algorithm,
        routing_mode="borderline_review", decision_confidence=0.70,
        borderline_score=0.90, explanation="probe")

    est = estimate_resources(decision, spec)
    hw = map_hardware(est, decision)
    print(f"{algorithm[:44]:<46} {est.estimated_memory_gb:>9.3f} GB  "
          f"{est.estimated_runtime_seconds:>9.2f} s  -> {hw.backend_type}")
    print(f"{'':48}{hw.device_spec}")

## 10. Save the results

Download the JSON and keep it alongside the thesis: it is the evidence
behind every calibrated constant, and it records which host produced them.

In [ ]:
import json

with open("/content/calibration_colab.json") as fh:
    data = json.load(fh)

print("host     :", data.get("host"))
print("host_tag :", data.get("host_tag"))
print()
for name, r in data.get("results", {}).items():
    if "memory_exp" in r:
        print(f"{name:<26} mem_exp={r['memory_exp']:.2f} "
              f"(R2={r['memory_r2']:.3f})  "
              f"rt_exp={r['runtime_exp']:.2f} (R2={r['runtime_r2']:.3f})")

from google.colab import files
files.download("/content/calibration_colab.json")

---

## What to do with these numbers

**Paste into `resource_constants.py`:** the exponents, and `base_memory_gb`.
Set `calibrated: True` and `calibration_host` to the printed host tag.

**Do not paste** `base_runtime_s` as if it described your laptop. Either
keep the uncalibrated value and leave `calibrated: False`, or re-run the
harness locally once Smart App Control or WSL is sorted and use those
figures for the laptop profile.

**Report honestly.** With this notebook you can say: scaling exponents
fitted on real measurements with R² reported; memory formula verified
against actual allocations; GPU speedup measured on a T4 at a stated input
size; multi-node MPI untested and out of scope. That is a defensible set of
claims, and far stronger than unattributed numbers.

**One test will now fail**, deliberately:
`test_calibrated_flag_is_false_until_benchmarked`. It exists to make you
consciously acknowledge a profile moved from reasoned to measured. Update
it and say so in the write-up.

Also note `test_calibrated_profiles_must_name_their_host` — it fails if a
profile claims `calibrated: True` without a `calibration_host`.